In [2]:
import csv
import heapq
import math
import time
from collections import defaultdict


def euclidean_distance(x1, y1, z1, x2, y2, z2):
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2 + (z1 - z2) ** 2)


coordinates = {}
adjacency_list = defaultdict(list)

# Read Coordinates
with open("Coordinates.csv", "r") as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:
        if row:
            star_name, x, y, z = (
                row[0].strip(),
                float(row[1]),
                float(row[2]),
                float(row[3]),
            )
            coordinates[star_name] = (x, y, z)

# Read Distances
with open("distances.csv", "r") as file:
    reader = csv.reader(file)
    for row in reader:
        if row:
            source, destination, dist = (
                row[0].strip(),
                row[1].strip(),
                float(row[2]),
            )
            adjacency_list[source].append((destination, dist))


def dijkstra(start_star, goal_star):
    start_time = time.perf_counter()
    priority_queue = [(0, start_star, [start_star])]
    visited = set()
    expanded_nodes = 0

    while priority_queue:
        cost, current, path = heapq.heappop(priority_queue)

        if current in visited:
            continue
        visited.add(current)
        expanded_nodes += 1

        if current == goal_star:
            end_time = time.perf_counter()
            return path, cost, expanded_nodes, (end_time - start_time) * 1000

        for neighbor, weight in adjacency_list.get(current, []):
            if neighbor not in visited:
                heapq.heappush(
                    priority_queue, (cost + weight, neighbor, path + [neighbor])
                )

    end_time = time.perf_counter()
    return None, float("inf"), expanded_nodes, (end_time - start_time) * 1000


def astar(start_star, goal_star):
    start_time = time.perf_counter()

    if goal_star not in coordinates or start_star not in coordinates:
        return None, float("inf"), 0, 0.0

    x2, y2, z2 = coordinates[goal_star]
    x1, y1, z1 = coordinates[start_star]
    h0 = euclidean_distance(x1, y1, z1, x2, y2, z2)

    priority_queue = [(h0, 0, start_star, [start_star])]
    visited = set()
    expanded_nodes = 0

    while priority_queue:
        f, g, current, path = heapq.heappop(priority_queue)

        if current in visited:
            continue
        visited.add(current)
        expanded_nodes += 1

        if current == goal_star:
            end_time = time.perf_counter()
            return path, g, expanded_nodes, (end_time - start_time) * 1000

        for neighbor, weight in adjacency_list.get(current, []):
            if neighbor not in visited:
                # Check if neighbor has coordinates for heuristic calculation
                if neighbor in coordinates:
                    nx, ny, nz = coordinates[neighbor]
                    h_new = euclidean_distance(nx, ny, nz, x2, y2, z2)
                else:
                    h_new = 0  # Fallback to Dijkstra behavior if coordinates missing

                g_new = g + weight
                f_new = g_new + h_new
                heapq.heappush(
                    priority_queue,
                    (f_new, g_new, neighbor, path + [neighbor]),
                )

    end_time = time.perf_counter()
    return None, float("inf"), expanded_nodes, (end_time - start_time) * 1000


# Test Execution
test_cases = [
    ("Sun", "Upsilon Andromedae"),
    ("Sun", "61 Virginis"),
    ("TRAPPIST-1", "55 Cancri"),
]

for src, dst in test_cases:
    print("=" * 50)
    print(f" Source: {src} ---> Destination: {dst}")
    print("=" * 50)

    d_path, d_cost, d_exp, d_time = dijkstra(src, dst)
    a_path, a_cost, a_exp, a_time = astar(src, dst)

    # Dijkstra Result
    if d_path is not None:
        print(f"  [Dijkstra] Path: {' -> '.join(d_path)}")
        print(
            f"             Cost: {d_cost:.2f} | Expanded Nodes: {d_exp} | Time: {d_time:.3f} ms"
        )
    else:
        print("  [Dijkstra] Path NOT found!")

    # A* Search Result
    if a_path is not None:
        print(f"  [A* Search] Path: {' -> '.join(a_path)}")
        print(
            f"              Cost: {a_cost:.2f} | Expanded Nodes: {a_exp} | Time: {a_time:.3f} ms"
        )
    else:
        print("  [A* Search] Path NOT found!")

    print()

 Source: Sun ---> Destination: Upsilon Andromedae
  [Dijkstra] Path NOT found!
  [A* Search] Path NOT found!

 Source: Sun ---> Destination: 61 Virginis
  [Dijkstra] Path: Sun -> Tau Ceti -> 61 Virginis
             Cost: 6362.00 | Expanded Nodes: 16 | Time: 0.053 ms
  [A* Search] Path: Sun -> Tau Ceti -> 61 Virginis
              Cost: 6362.00 | Expanded Nodes: 6 | Time: 0.051 ms

 Source: TRAPPIST-1 ---> Destination: 55 Cancri
  [Dijkstra] Path: TRAPPIST-1 -> 55 Cancri
             Cost: 26060.00 | Expanded Nodes: 19 | Time: 0.064 ms
  [A* Search] Path: TRAPPIST-1 -> 55 Cancri
              Cost: 26060.00 | Expanded Nodes: 3 | Time: 0.026 ms

